# SWaT RAG explanation layer, four detectors

Reuses the data loading, labelling, attack-disjoint split, evaluation utilities, feature
engineering, Granger sensor graph, forecaster windowing, and the four model classes (Random
Forest, GDN, T-GCN forecaster, T-GCN classifier) from swat_final.ipynb. New here: an
attack-pattern library built from iTrust's official Data Collection at SWaT Testbed, 20 July
2019 document, a Neo4j knowledge graph over sensors, the Granger graph, process stages, and
attack patterns, subgraph retrieval keyed on a detector's top implicated sensors, and
natural-language explanation generation grounded in that subgraph via the Groq API. Adapted
from the BATADAL version of this pipeline to SWaT's single Granger graph and four detectors.

Runs entirely on CPU. The two forecasters are small models over 15k rows, and the knowledge
graph and LLM calls are I/O-bound rather than compute-bound.


## 0. Setup

Credentials are entered via getpass at runtime rather than hardcoded. You will need:
- A free Neo4j AuraDB instance (or local Neo4j Desktop/Docker), its URI, username, and password.
- A Groq API key.


In [1]:
!pip install -q neo4j groq

In [2]:
import os

os.environ["GROQ_API_KEY"] = "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"
os.environ["NEO4J_URI"] = "neo4j+s://156b70ec.databases.neo4j.io"
os.environ["NEO4J_USERNAME"] = "156b70ec"
os.environ["NEO4J_PASSWORD"] = "o4l3vjwcVMN5EfbpnJa4_VI218ZOE22iddOvQVrv4sE"
os.environ["NEO4J_DATABASE"] = "156b70ec"

In [3]:
import getpass, os

NEO4J_URI = os.environ.get("NEO4J_URI") 
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME") 
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD") 
NEO4J_DATABASE = os.environ.get("NEO4J_DATABASE") 
GROQ_API_KEY = os.environ.get("GROQ_API_KEY") 

RESET_DATASET = True   # wipe any previous 'swat' nodes on this run, so re-running is idempotent


## 1. Data and labels

Timestamps are parsed, the first minute of the run
is dropped as a settling period, `ATTACK_ID` is 0 for normal rows and 1-6 for the six documented
attacks, converted from Singapore time to UTC.

In [4]:
import itertools
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
)
from imblearn.over_sampling import SMOTE
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.stats.multitest import multipletests
import warnings

FILE_PATH_XLSX = "SWaTdatasets/SWaT_dataset_Jul 19 v2.xlsx"
FILE_PATH_CSV = "SWaTdatasets/SWaT_dataset_Jul19_v2.csv"
HEADER_ROW = 1
OUT_DIR = "SWaTdatasets"
TS_COL = "GMT +0"
RANDOM_STATE = 42

ATTACK_WINDOWS = [
    ('2019-07-20 15:08:46', '2019-07-20 15:10:31'),  # FIT401 spoof
    ('2019-07-20 15:15:00', '2019-07-20 15:19:32'),  # LIT301 spoof
    ('2019-07-20 15:26:57', '2019-07-20 15:30:48'),  # P601 OFF->ON
    ('2019-07-20 15:38:50', '2019-07-20 15:46:20'),  # MV201+P101 multi-point
    ('2019-07-20 15:54:00', '2019-07-20 15:56:00'),  # MV501 OPEN->CLOSE -- corrected to official iTrust times
    ('2019-07-20 16:02:56', '2019-07-20 16:16:18'),  # P301 ON->OFF -- corrected to official iTrust times
]

ROLL_WINDOW = 10
WINDOW_GDN = 12
WINDOW_TGCN = 12
STRIDE = 1
CALIB_MINUTES = 5

GRANGER_MAXLAG = 3
GRANGER_ALPHA = 0.05
GRANGER_SUBSAMPLE = 5

EPOCHS = 30
BATCH_SIZE = 128
LR = 1e-3
HIDDEN = 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print("Device:", DEVICE)


Device: cpu


In [5]:
def load_and_label():
    if not os.path.exists(FILE_PATH_CSV):
        raw = pd.read_excel(FILE_PATH_XLSX, header=HEADER_ROW)
        raw.columns = [str(c).strip() for c in raw.columns]
        os.makedirs(os.path.dirname(FILE_PATH_CSV), exist_ok=True)
        raw.to_csv(FILE_PATH_CSV, index=False)
    df = pd.read_csv(FILE_PATH_CSV, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]

    df[TS_COL] = pd.to_datetime(df[TS_COL], errors='coerce', utc=True)
    df = df.dropna(subset=[TS_COL]).sort_values(TS_COL).reset_index(drop=True)

    plant_start = df[TS_COL].min()
    df = df[df[TS_COL] >= plant_start + pd.Timedelta(minutes=1)].reset_index(drop=True)
    df = df.ffill().bfill()

    df['label'] = 0
    df['ATTACK_ID'] = 0
    for i, (start, end) in enumerate(ATTACK_WINDOWS, start=1):
        start_ts = pd.to_datetime(start).tz_localize('Asia/Singapore').tz_convert('UTC')
        end_ts = pd.to_datetime(end).tz_localize('Asia/Singapore').tz_convert('UTC')
        mask = (df[TS_COL] >= start_ts) & (df[TS_COL] <= end_ts)
        df.loc[mask, 'label'] = 1
        df.loc[mask, 'ATTACK_ID'] = i

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c not in ('label', 'ATTACK_ID')]
    const_cols = df[num_cols].nunique()[lambda s: s <= 1].index.tolist()
    if const_cols:
        df = df.drop(columns=const_cols)

    non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
    non_numeric = [c for c in non_numeric if c != TS_COL]
    for c in non_numeric:
        df[c] = df[c].astype('category').cat.codes

    feature_cols = [c for c in df.columns if c not in (TS_COL, 'label', 'ATTACK_ID')]
    print("Shape:", df.shape, "| features:", len(feature_cols))
    print("Attack ratio:", round(df['label'].mean(), 4), f"({df['label'].sum()} of {len(df)} rows)")
    return df, feature_cols


df, FEATURE_COLS = load_and_label()
df.head()


C:\Users\MADHU\AppData\Local\Temp\ipykernel_10372\1775269468.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[TS_COL] = pd.to_datetime(df[TS_COL], errors='coerce', utc=True)


Shape: (14936, 80) | features: 77
Attack ratio: 0.1326 (1981 of 14936 rows)


,GMT +0,FIT 101,LIT 101,MV 101,P1_STATE,P101 Status,P102 Status,AIT 201,AIT 202,AIT 203,...,LSH 603,LSL 601,LSL 602,LSL 603,P6 STATE,P601 Status,P602 Status,P603 Status,label,ATTACK_ID
0,2019-07-20 04:31:00.004013+00:00,0,2958,1,1,1,0,230,1066,0,...,0,0,0,0,0,0,0,0,0,0
1,2019-07-20 04:31:01.004013+00:00,0,2956,1,1,1,0,230,1066,1,...,0,0,0,0,0,0,0,0,0,0
2,2019-07-20 04:31:02.002014100+00:00,0,2949,1,1,1,0,230,1066,2,...,0,0,0,0,0,0,0,0,0,0
3,2019-07-20 04:31:03.004013+00:00,0,2946,1,1,1,0,230,1066,3,...,0,0,0,0,0,0,0,0,0,0
4,2019-07-20 04:31:04.004013+00:00,0,2937,1,1,1,0,230,1066,4,...,0,0,0,0,0,0,0,0,0,0


## 2. Split

Train on attacks 1-4 plus preceding normal data, test
on attack 6 plus its own five-minute pre-attack calibration slice, hold attack 5 (`val5`) out
entirely from training and use it only to select a calibration rule.

In [6]:
def get_attack_bounds_utc():
    bounds = []
    for start, end in ATTACK_WINDOWS:
        s = pd.to_datetime(start).tz_localize('Asia/Singapore').tz_convert('UTC')
        e = pd.to_datetime(end).tz_localize('Asia/Singapore').tz_convert('UTC')
        bounds.append((s, e))
    return bounds


attack_bounds = get_attack_bounds_utc()
ATTACK5_START, ATTACK5_END = attack_bounds[4]
ATTACK6_START, ATTACK6_END = attack_bounds[5]
CALIB5_START = ATTACK5_START - pd.Timedelta(minutes=CALIB_MINUTES)
CALIB_START = ATTACK6_START - pd.Timedelta(minutes=CALIB_MINUTES)

row_idx = np.arange(len(df))
y_all = df['label'].values
attack_id_all = df['ATTACK_ID'].values

ts_vals = df[TS_COL].values
in_calib5_window = (ts_vals >= np.datetime64(CALIB5_START)) & (ts_vals < np.datetime64(ATTACK5_START))

train_idx = row_idx[(attack_id_all <= 4) & (ts_vals < np.datetime64(CALIB_START)) & (~in_calib5_window)]
test_idx = row_idx[(ts_vals >= np.datetime64(CALIB_START)) & (ts_vals <= np.datetime64(ATTACK6_END))]
val5_idx = row_idx[(ts_vals >= np.datetime64(CALIB5_START)) & (ts_vals <= np.datetime64(ATTACK5_END))]

print(f"train: {len(train_idx)} rows ({y_all[train_idx].sum()} attack rows, attacks 1-4)")
print(f"test:  {len(test_idx)} rows ({y_all[test_idx].sum()} attack rows, attack 6 plus calibration slice)")
print(f"val5:  {len(val5_idx)} rows ({y_all[val5_idx].sum()} attack rows, attack 5 plus calibration slice)")


C:\Users\MADHU\AppData\Local\Temp\ipykernel_10372\1706626789.py:21: UserWarning: no explicit representation of timezones available for np.datetime64
  in_calib5_window = (ts_vals >= np.datetime64(CALIB5_START)) & (ts_vals < np.datetime64(ATTACK5_START))
C:\Users\MADHU\AppData\Local\Temp\ipykernel_10372\1706626789.py:23: UserWarning: no explicit representation of timezones available for np.datetime64
  train_idx = row_idx[(attack_id_all <= 4) & (ts_vals < np.datetime64(CALIB_START)) & (~in_calib5_window)]
C:\Users\MADHU\AppData\Local\Temp\ipykernel_10372\1706626789.py:24: UserWarning: no explicit representation of timezones available for np.datetime64
  test_idx = row_idx[(ts_vals >= np.datetime64(CALIB_START)) & (ts_vals <= np.datetime64(ATTACK6_END))]
C:\Users\MADHU\AppData\Local\Temp\ipykernel_10372\1706626789.py:25: UserWarning: no explicit representation of timezones available for np.datetime64
  val5_idx = row_idx[(ts_vals >= np.datetime64(CALIB5_START)) & (ts_vals <= np.datetime6

train: 11992 rows (1059 attack rows, attacks 1-4)
test:  1102 rows (802 attack rows, attack 6 plus calibration slice)
val5:  420 rows (120 attack rows, attack 5 plus calibration slice)


## 3. Evaluation utilities

In [7]:
def point_adjust(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int).copy()
    anomaly_state = False
    for i in range(len(y_true)):
        if y_true[i] == 1 and y_pred[i] == 1 and not anomaly_state:
            anomaly_state = True
            for j in range(i, 0, -1):
                if y_true[j] == 0:
                    break
                if y_pred[j] == 0:
                    y_pred[j] = 1
        elif y_true[i] == 0:
            anomaly_state = False
        if anomaly_state:
            y_pred[i] = 1
    return y_pred


def metrics_at_threshold(y_true, scores, thr, use_point_adjust=True):
    pred = (np.asarray(scores) >= thr).astype(int)
    pred_eval = point_adjust(y_true, pred) if use_point_adjust else pred
    return {
        "accuracy": accuracy_score(y_true, pred_eval),
        "f1": f1_score(y_true, pred_eval, zero_division=0),
        "precision": precision_score(y_true, pred_eval, zero_division=0),
        "recall": recall_score(y_true, pred_eval, zero_division=0),
        "roc_auc": roc_auc_score(y_true, scores) if len(set(y_true)) > 1 else np.nan,
        "pr_auc": average_precision_score(y_true, scores) if len(set(y_true)) > 1 else np.nan,
    }


def full_report(name, y_true, scores, thr):
    pa = metrics_at_threshold(y_true, scores, thr, use_point_adjust=True)
    raw = metrics_at_threshold(y_true, scores, thr, use_point_adjust=False)
    print(f"{name}, point-adjusted:", {k: round(v, 3) for k, v in pa.items()})
    print(f"{name}, raw:           ", {k: round(v, 3) for k, v in raw.items()})
    return pa, raw


def threshold_from_rule(rule, calib_scores, all_scores):
    if rule == "mean3std":
        thr = calib_scores.mean() + 3 * calib_scores.std()
    else:
        pct = int(rule.replace("pct", ""))
        thr = np.percentile(calib_scores, pct)
    return min(thr, np.percentile(all_scores, 99)) if np.isfinite(thr) else np.percentile(all_scores, 99)


CALIBRATION_RULES = ["mean3std", "pct75", "pct90", "pct95"]


def select_rule_via_attack5(val5_scores, val5_calib_scores, val5_labels, min_rows=10):
    if len(val5_calib_scores) < min_rows:
        return "mean3std", None
    best_rule, best_f1 = None, -1
    for rule in CALIBRATION_RULES:
        thr = threshold_from_rule(rule, val5_calib_scores, val5_scores)
        pred = (np.asarray(val5_scores) >= thr).astype(int)
        f1 = f1_score(val5_labels, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_rule = f1, rule
    return best_rule, best_f1


def calibrate_threshold_rowwise(scores, ts_series, rule):
    calib_mask = (ts_series.values >= np.datetime64(CALIB_START)) & (ts_series.values < np.datetime64(ATTACK6_START))
    calib_scores = np.asarray(scores)[calib_mask]
    if calib_mask.sum() < 10:
        return np.percentile(scores, 95)
    return threshold_from_rule(rule, calib_scores, scores)


def calibrate_threshold_windowed(scores, ts_series, window, rule):
    n = len(scores)
    if n == 0:
        return 0.0
    aligned_ts = pd.to_datetime(ts_series.values[window - 1: window - 1 + n], utc=True)
    calib_mask = np.asarray((aligned_ts >= CALIB_START) & (aligned_ts < ATTACK6_START))
    calib_scores = np.asarray(scores)[calib_mask]
    if calib_mask.sum() < 10:
        return np.percentile(scores, 95)
    return threshold_from_rule(rule, calib_scores, scores)


## 4. Random Forest

In [8]:
def add_rolling_features(data, cols, window):
    roll = data[cols].rolling(window, min_periods=1)
    feats = pd.concat([
        roll.mean().add_suffix('_mean'),
        roll.std().add_suffix('_std').fillna(0),
        roll.min().add_suffix('_min'),
        roll.max().add_suffix('_max'),
        data[cols].diff().fillna(0).add_suffix('_delta'),
    ], axis=1)
    return feats


roll_feats_all = add_rolling_features(df, FEATURE_COLS, ROLL_WINDOW)
X_all = pd.concat([df[FEATURE_COLS], roll_feats_all], axis=1).replace([np.inf, -np.inf], 0).fillna(0).values
print("Feature matrix shape:", X_all.shape)

scaler_rf = StandardScaler().fit(X_all[train_idx])
X_train_rf = scaler_rf.transform(X_all[train_idx])
X_test_rf = scaler_rf.transform(X_all[test_idx])
y_train_rf, y_test_rf = y_all[train_idx], y_all[test_idx]

rf = RandomForestClassifier(
    n_estimators=500, min_samples_leaf=1, max_features="sqrt",
    class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train_rf, y_train_rf)
test_probs_rf = rf.predict_proba(X_test_rf)[:, 1]

X_val5_rf = scaler_rf.transform(X_all[val5_idx])
y_val5_rf = y_all[val5_idx]
val5_probs_rf = rf.predict_proba(X_val5_rf)[:, 1]

ts_val5_rf = df.iloc[val5_idx][TS_COL]
calib5_mask_rf = (ts_val5_rf.values >= np.datetime64(CALIB5_START)) & (ts_val5_rf.values < np.datetime64(ATTACK5_START))
calib5_scores_rf = val5_probs_rf[calib5_mask_rf]

rf_rule, rf_rule_f1 = select_rule_via_attack5(val5_probs_rf, calib5_scores_rf, y_val5_rf)
print(f"Random Forest calibration rule selected via attack 5: {rf_rule} (attack 5 raw F1={rf_rule_f1})")

ts_test = df.iloc[test_idx][TS_COL]
rf_threshold = calibrate_threshold_rowwise(test_probs_rf, ts_test, rf_rule)
rf_results_pa, rf_results_raw = full_report("Random Forest", y_test_rf, test_probs_rf, rf_threshold)


Feature matrix shape: (14936, 462)
Random Forest calibration rule selected via attack 5: pct75 (attack 5 raw F1=0.11214953271028037)
Random Forest, point-adjusted: {'accuracy': 0.931, 'f1': 0.955, 'precision': 0.913, 'recall': 1.0, 'roc_auc': 0.58, 'pr_auc': 0.806}
Random Forest, raw:            {'accuracy': 0.435, 'f1': 0.45, 'precision': 0.77, 'recall': 0.318, 'roc_auc': 0.58, 'pr_auc': 0.806}


C:\Users\MADHU\AppData\Local\Temp\ipykernel_10372\511381057.py:34: UserWarning: no explicit representation of timezones available for np.datetime64
  calib5_mask_rf = (ts_val5_rf.values >= np.datetime64(CALIB5_START)) & (ts_val5_rf.values < np.datetime64(ATTACK5_START))
C:\Users\MADHU\AppData\Local\Temp\ipykernel_10372\899841276.py:67: UserWarning: no explicit representation of timezones available for np.datetime64
  calib_mask = (ts_series.values >= np.datetime64(CALIB_START)) & (ts_series.values < np.datetime64(ATTACK6_START))


## 5. Sensor graph and asset typing

`node_type_for_tag`
and `stage_for_tag` classify each sensor by SWaT's own tag convention (leading digit = process
stage P1-P6, letters = instrument type: FIT flow, LIT level, AIT analyzer, PIT pressure, MV
motorised valve, P pump, UV UV-dechlorinator), giving the knowledge graph the same kind of
asset-type and physical-process structure that the BATADAL graph gets from `plc_edges`.

In [9]:
def build_granger_adjacency(normal_scaled, maxlag=GRANGER_MAXLAG, alpha=GRANGER_ALPHA, subsample=GRANGER_SUBSAMPLE,
                             verbose=True):
    n = normal_scaled.shape[1]
    data = normal_scaled[::subsample] if subsample > 1 else normal_scaled

    pairs, pvals = [], []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for cause, effect in itertools.permutations(range(n), 2):
            x = data[:, [effect, cause]]
            try:
                res = grangercausalitytests(x, maxlag=[maxlag])
                p = res[maxlag][0]['ssr_ftest'][1]
            except Exception:
                p = 1.0
            pairs.append((cause, effect))
            pvals.append(p)

    reject, _, _, _ = multipletests(pvals, alpha=alpha, method='fdr_bh')
    adj = np.zeros((n, n), dtype=np.float32)
    for (cause, effect), rej in zip(pairs, reject):
        if rej:
            adj[cause, effect] = 1.0
    np.fill_diagonal(adj, 1.0)

    n_edges = int(adj.sum()) - n
    if verbose:
        print(f"Granger graph, lag={maxlag}: {n_edges} directed edges out of {n * (n - 1)} ordered pairs "
              f"tested (FDR alpha={alpha})")
    return adj


sensor_cols = [c for c in FEATURE_COLS if df[c].nunique() > 1]
n_nodes = len(sensor_cols)

normal_train = df.iloc[train_idx]
normal_train = normal_train[normal_train['label'] == 0]
scaler_g = StandardScaler().fit(normal_train[sensor_cols])
normal_scaled_g = scaler_g.transform(normal_train[sensor_cols])

adj = build_granger_adjacency(normal_scaled_g)
edge_index = torch.tensor(np.array(np.nonzero(adj)), dtype=torch.long).to(DEVICE)

adj_t = adj.T
deg = adj_t.sum(axis=1, keepdims=True)
deg[deg == 0] = 1.0
adj_norm = torch.tensor(adj_t / deg, dtype=torch.float32).to(DEVICE)


Granger graph, lag=3: 302 directed edges out of 1892 ordered pairs tested (FDR alpha=0.05)


In [10]:
INSTRUMENT_TYPES = {
    "FIT": "flow sensor", "LIT": "level sensor", "AIT": "analyzer",
    "PIT": "pressure sensor", "DPIT": "differential pressure sensor",
    "MV": "motorised valve", "UV": "UV dechlorinator", "P": "pump", "FCV": "flow control valve",
}
STAGE_NAMES = {
    1: "P1 raw water intake", 2: "P2 chemical dosing", 3: "P3 ultrafiltration",
    4: "P4 UV dechlorination", 5: "P5 reverse osmosis", 6: "P6 RO permeate/backwash",
}

def stage_for_tag(tag):
    for ch in tag:
        if ch.isdigit():
            return int(ch)
    return 0

def node_type_for_tag(tag):
    for prefix in sorted(INSTRUMENT_TYPES, key=len, reverse=True):
        if tag.upper().startswith(prefix):
            return INSTRUMENT_TYPES[prefix]
    return "sensor"

ASSET_TYPES = {tag: node_type_for_tag(tag) for tag in sensor_cols}
ASSET_STAGES = {tag: stage_for_tag(tag) for tag in sensor_cols}
print(pd.Series(ASSET_TYPES).value_counts())
print(pd.Series(ASSET_STAGES).value_counts().sort_index())


analyzer                        11
flow sensor                      9
pump                             8
motorised valve                  7
level sensor                     3
pressure sensor                  3
differential pressure sensor     1
UV dechlorinator                 1
sensor                           1
Name: count, dtype: int64
1     5
2     7
3    12
4     5
5    12
6     3
Name: count, dtype: int64


## 6. Forecaster windowing

In [11]:
def make_windows(data, labels, window, stride):
    Xs, ys = [], []
    for i in range(0, len(data) - window, stride):
        Xs.append(data[i:i + window])
        ys.append(int(labels[i + window - 1]))
    if not Xs:
        return np.empty((0, window, data.shape[1])), np.empty((0,))
    return np.array(Xs), np.array(ys)


df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)
df_val5 = df.iloc[val5_idx].reset_index(drop=True)

train_scaled_g = scaler_g.transform(df_train[sensor_cols])
test_scaled_g = scaler_g.transform(df_test[sensor_cols])
val5_scaled_g = scaler_g.transform(df_val5[sensor_cols])
normal_mask_train = df_train['label'].values == 0


def build_forecast_windows(window):
    Xw_train, _ = make_windows(train_scaled_g[normal_mask_train], df_train['label'].values[normal_mask_train],
                                window, STRIDE)
    Xw_test, yw_test = make_windows(test_scaled_g, df_test['label'].values, window, STRIDE)
    Xw_val5, yw_val5 = make_windows(val5_scaled_g, df_val5['label'].values, window, STRIDE)
    return Xw_train, Xw_test, yw_test, Xw_val5, yw_val5


def select_forecaster_rule(model, edge_or_adj, err_mean, err_std, Xw_val5, yw_val5, window):
    if len(Xw_val5) == 0:
        return "mean3std", None
    val5_scores, val5_labels = score_forecaster(model, edge_or_adj, Xw_val5, yw_val5, err_mean, err_std)
    aligned_ts = pd.to_datetime(df_val5[TS_COL].values[window - 1: window - 1 + len(val5_scores)], utc=True)
    calib_mask = np.asarray((aligned_ts >= CALIB5_START) & (aligned_ts < ATTACK5_START))
    calib_scores = val5_scores[calib_mask]
    return select_rule_via_attack5(val5_scores, calib_scores, val5_labels)


## 7. GDN

In [12]:
class GDN(nn.Module):
    def __init__(self, n_nodes, hidden):
        super().__init__()
        self.n_nodes = n_nodes
        self.hidden = hidden
        self.node_embed = nn.Parameter(torch.randn(n_nodes, hidden))
        self.hist_enc = nn.GRU(1, hidden, batch_first=True)
        self.attn = nn.Linear(2 * hidden, 1)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x, edge_index):
        history = x[:, :-1, :]
        target = x[:, -1, :]
        b, t, n = history.shape

        hist_in = history.permute(0, 2, 1).reshape(b * n, t, 1)
        _, hn = self.hist_enc(hist_in)
        h = hn.squeeze(0).reshape(b, n, self.hidden)

        src, dst = edge_index
        e_src = self.node_embed[src].unsqueeze(0).expand(b, -1, -1)
        e_dst = self.node_embed[dst].unsqueeze(0).expand(b, -1, -1)
        alpha = torch.softmax(self.attn(torch.cat([e_src, e_dst], dim=-1)).squeeze(-1), dim=-1)

        agg = torch.zeros(b, n, self.hidden, device=x.device)
        agg.index_add_(1, dst, alpha.unsqueeze(-1) * h[:, src, :])

        pred = self.out(agg).squeeze(-1)
        per_node_err = (pred - target).pow(2)
        score = per_node_err.mean(dim=1)
        return score, per_node_err


def train_forecaster(model, edge_or_adj, Xw_train, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE):
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(Xw_train, dtype=torch.float32)),
        batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            opt.zero_grad()
            score, _ = model(xb, edge_or_adj)
            loss = score.mean()
            loss.backward()
            opt.step()
    return model, loader


def score_forecaster(model, edge_or_adj, Xw, yw, err_mean, err_std, batch_size=BATCH_SIZE):
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(Xw, dtype=torch.float32), torch.tensor(yw, dtype=torch.float32)),
        batch_size=batch_size, shuffle=False)
    model.eval()
    scores, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            _, per_node_err = model(xb, edge_or_adj)
            z = (per_node_err.cpu().numpy() - err_mean) / err_std
            scores.append(z.max(axis=1))
            labels.append(yb.numpy())
    return np.concatenate(scores), np.concatenate(labels)


def fit_error_stats(model, edge_or_adj, loader):
    model.eval()
    errs = []
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            _, per_node_err = model(xb, edge_or_adj)
            errs.append(per_node_err.cpu().numpy())
    errs = np.concatenate(errs)
    return errs.mean(axis=0), errs.std(axis=0) + 1e-6


torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

Xw_train_gdn, Xw_test_gdn, yw_test_gdn, Xw_val5_gdn, yw_val5_gdn = build_forecast_windows(WINDOW_GDN)
gdn = GDN(n_nodes, HIDDEN).to(DEVICE)
gdn, gdn_train_loader = train_forecaster(gdn, edge_index, Xw_train_gdn)
gdn_err_mean, gdn_err_std = fit_error_stats(gdn, edge_index, gdn_train_loader)

gdn_rule, gdn_rule_f1 = select_forecaster_rule(gdn, edge_index, gdn_err_mean, gdn_err_std, Xw_val5_gdn, yw_val5_gdn, WINDOW_GDN)
print(f"GDN calibration rule selected via attack 5: {gdn_rule} (attack 5 raw F1={gdn_rule_f1})")

gdn_test_scores, gdn_test_labels = score_forecaster(gdn, edge_index, Xw_test_gdn, yw_test_gdn, gdn_err_mean, gdn_err_std)
gdn_threshold = calibrate_threshold_windowed(gdn_test_scores, df_test[TS_COL], WINDOW_GDN, gdn_rule)
gdn_results_pa, gdn_results_raw = full_report("GDN", gdn_test_labels, gdn_test_scores, gdn_threshold)


GDN calibration rule selected via attack 5: pct75 (attack 5 raw F1=0.41025641025641024)
GDN, point-adjusted: {'accuracy': 0.933, 'f1': 0.956, 'precision': 0.916, 'recall': 1.0, 'roc_auc': 0.684, 'pr_auc': 0.858}
GDN, raw:            {'accuracy': 0.738, 'f1': 0.804, 'precision': 0.89, 'recall': 0.734, 'roc_auc': 0.684, 'pr_auc': 0.858}


## 8. T-GCN forecaster and classifier

In [13]:
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj_norm):
        agg = torch.einsum('ij,bjd->bid', adj_norm, x)
        return torch.relu(self.lin(agg))


class TGCN(nn.Module):
    def __init__(self, n_nodes, hidden):
        super().__init__()
        self.gcn = GCNLayer(1, hidden)
        self.gru = nn.GRU(hidden * n_nodes, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_nodes)
        self.n_nodes = n_nodes

    def forward(self, x, adj_norm):
        history = x[:, :-1, :]
        target = x[:, -1, :]
        b, t, n = history.shape
        gcn_out = []
        for step in range(t):
            xt = history[:, step, :].unsqueeze(-1)
            h = self.gcn(xt, adj_norm)
            gcn_out.append(h.reshape(b, -1))
        seq = torch.stack(gcn_out, dim=1)
        _, hn = self.gru(seq)
        pred = self.out(hn.squeeze(0))
        per_node_err = (pred - target).pow(2)
        score = per_node_err.mean(dim=1)
        return score, per_node_err


torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

Xw_train_tgcn, Xw_test_tgcn, yw_test_tgcn, Xw_val5_tgcn, yw_val5_tgcn = build_forecast_windows(WINDOW_TGCN)
tgcn = TGCN(n_nodes, HIDDEN).to(DEVICE)
tgcn, tgcn_train_loader = train_forecaster(tgcn, adj_norm, Xw_train_tgcn)
tgcn_err_mean, tgcn_err_std = fit_error_stats(tgcn, adj_norm, tgcn_train_loader)

tgcn_test_scores, tgcn_test_labels = score_forecaster(tgcn, adj_norm, Xw_test_tgcn, yw_test_tgcn, tgcn_err_mean, tgcn_err_std)

tgcn_rule, tgcn_rule_f1 = select_forecaster_rule(tgcn, adj_norm, tgcn_err_mean, tgcn_err_std, Xw_val5_tgcn, yw_val5_tgcn, WINDOW_TGCN)
print(f"T-GCN forecaster calibration rule selected via attack 5: {tgcn_rule} (attack 5 raw F1={tgcn_rule_f1})")

tgcn_threshold = calibrate_threshold_windowed(tgcn_test_scores, df_test[TS_COL], WINDOW_TGCN, tgcn_rule)
tgcn_results_pa, tgcn_results_raw = full_report("T-GCN forecaster", tgcn_test_labels, tgcn_test_scores, tgcn_threshold)


T-GCN forecaster calibration rule selected via attack 5: pct75 (attack 5 raw F1=0.4066390041493776)
T-GCN forecaster, point-adjusted: {'accuracy': 0.933, 'f1': 0.956, 'precision': 0.916, 'recall': 1.0, 'roc_auc': 0.842, 'pr_auc': 0.906}
T-GCN forecaster, raw:            {'accuracy': 0.743, 'f1': 0.809, 'precision': 0.891, 'recall': 0.742, 'roc_auc': 0.842, 'pr_auc': 0.906}


In [14]:
class TGCNClassifier(nn.Module):
    def __init__(self, n_nodes, hidden):
        super().__init__()
        self.gcn = GCNLayer(1, hidden)
        self.gru = nn.GRU(hidden * n_nodes, hidden, batch_first=True)
        self.classifier = nn.Linear(hidden, 1)
        self.n_nodes = n_nodes

    def forward(self, x, adj_norm):
        b, t, n = x.shape
        gcn_out = []
        for step in range(t):
            xt = x[:, step, :].unsqueeze(-1)
            h = self.gcn(xt, adj_norm)
            gcn_out.append(h.reshape(b, -1))
        seq = torch.stack(gcn_out, dim=1)
        _, hn = self.gru(seq)
        return self.classifier(hn.squeeze(0)).squeeze(-1)


Xw_train_full, yw_train_full = make_windows(train_scaled_g, df_train['label'].values, WINDOW_TGCN, STRIDE)
n_train_windows, win_len, _ = Xw_train_full.shape
X_flat = Xw_train_full.reshape(n_train_windows, -1)

minority_n = int(yw_train_full.sum())
k_neighbors = max(1, min(5, minority_n - 1))
smote = SMOTE(sampling_strategy='auto', k_neighbors=k_neighbors, random_state=RANDOM_STATE)
X_flat_bal, y_train_bal = smote.fit_resample(X_flat, yw_train_full)
X_train_bal = X_flat_bal.reshape(-1, win_len, n_nodes)
print(f"SMOTE: train {n_train_windows} -> {len(y_train_bal)} windows (k_neighbors={k_neighbors})")

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

tgcn_clf = TGCNClassifier(n_nodes, HIDDEN).to(DEVICE)
opt = torch.optim.Adam(tgcn_clf.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()
clf_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_train_bal, dtype=torch.float32), torch.tensor(y_train_bal, dtype=torch.float32)),
    batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(EPOCHS):
    tgcn_clf.train()
    for xb, yb in clf_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(tgcn_clf(xb, adj_norm), yb)
        loss.backward()
        opt.step()


def score_classifier(model, Xw, yw, batch_size=BATCH_SIZE):
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(Xw, dtype=torch.float32), torch.tensor(yw, dtype=torch.float32)),
        batch_size=batch_size, shuffle=False)
    model.eval()
    scores, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            prob = torch.sigmoid(model(xb, adj_norm))
            scores.append(prob.cpu().numpy())
            labels.append(yb.numpy())
    return np.concatenate(scores), np.concatenate(labels)


Xw_test_clf, yw_test_clf = make_windows(test_scaled_g, df_test['label'].values, WINDOW_TGCN, STRIDE)
tgcn_clf_test_scores, tgcn_clf_test_labels = score_classifier(tgcn_clf, Xw_test_clf, yw_test_clf)

tgcn_clf_val5_scores, tgcn_clf_val5_labels = score_classifier(tgcn_clf, Xw_val5_tgcn, yw_val5_tgcn)
aligned_ts_val5_clf = pd.to_datetime(
    df_val5[TS_COL].values[WINDOW_TGCN - 1: WINDOW_TGCN - 1 + len(tgcn_clf_val5_scores)], utc=True
)
calib5_mask_clf = np.asarray((aligned_ts_val5_clf >= CALIB5_START) & (aligned_ts_val5_clf < ATTACK5_START))
calib5_scores_clf = tgcn_clf_val5_scores[calib5_mask_clf]

tgcn_clf_rule, tgcn_clf_rule_f1 = select_rule_via_attack5(tgcn_clf_val5_scores, calib5_scores_clf, tgcn_clf_val5_labels)
print(f"T-GCN classifier calibration rule selected via attack 5: {tgcn_clf_rule} (attack 5 raw F1={tgcn_clf_rule_f1})")

tgcn_clf_threshold = calibrate_threshold_windowed(tgcn_clf_test_scores, df_test[TS_COL], WINDOW_TGCN, tgcn_clf_rule)
tgcn_clf_results_pa, tgcn_clf_results_raw = full_report("T-GCN classifier", tgcn_clf_test_labels, tgcn_clf_test_scores, tgcn_clf_threshold)


SMOTE: train 11980 -> 21842 windows (k_neighbors=5)
T-GCN classifier calibration rule selected via attack 5: pct75 (attack 5 raw F1=0.3157894736842105)
T-GCN classifier, point-adjusted: {'accuracy': 0.933, 'f1': 0.956, 'precision': 0.916, 'recall': 1.0, 'roc_auc': 0.61, 'pr_auc': 0.766}
T-GCN classifier, raw:            {'accuracy': 0.283, 'f1': 0.192, 'precision': 0.56, 'recall': 0.116, 'roc_auc': 0.61, 'pr_auc': 0.766}


## 9. Attack pattern library

Built from iTrust's official *Data Collection at SWaT Testbed, 20 July
2019* PDF, which is the ground-truth source for what each of the six attacks targeted and its
intended physical effect — the SWaT analogue of BATADAL's `ATTACK_DESCRIPTIONS`. Attribution
uses each sensor's raw-value deviation from its own normal-operation statistics (median and MAD,
calibrated on the training partition's normal rows), independent of any detector's own scoring, so
this library stays usable for attributing all four detectors on equal footing.

In [15]:
ATTACK_DESCRIPTIONS = {
    1: "Attack on FIT401: spoofed flow reading from 0.8 to 0.5, intended to stop de-chlorination "
       "by switching off UV401. Targets stage P4.",
    2: "Attack on LIT301: spoofed tank level from 835 to 1024, intended to eventually cause "
       "underflow in tank T301. Targets stage P3.",
    3: "Attack on P601: pump switched from OFF to ON, intended to increase water level in the raw "
       "water tank. Targets stage P6/P1.",
    4: "Multi-point attack on MV201 (CLOSE to OPEN) and P101 (OFF to ON), intended to overflow "
       "tank T301. Targets stages P1/P2/P3.",
    5: "Attack on MV501: valve switched from OPEN to CLOSE, intended to drain water from the "
       "reverse-osmosis unit. Targets stage P5.",
    6: "Attack on P301: pump switched from ON to OFF, intended to halt the ultrafiltration (stage "
       "3) process. Targets stage P3.",
}
ATTACK_PRIMARY_SENSOR = {1: "FIT401", 2: "LIT301", 3: "P601", 4: "MV201", 5: "MV501", 6: "P301"}
ATTACK_SECONDARY_SENSOR = {4: "P101"}


def calibrate_robust(values):
    median = np.median(values, axis=0)
    mad = np.median(np.abs(values - median), axis=0) + 1e-6
    return median, mad


def zscore(values, median, mad):
    return (values - median) / mad


def top_k_sensors(z_matrix, k=5):
    mean_z = np.abs(z_matrix).mean(axis=0)
    top_idx = np.argsort(-mean_z)[:k]
    return [(sensor_cols[j], float(mean_z[j])) for j in top_idx]


normal_median, normal_mad = calibrate_robust(normal_train[sensor_cols].values)

attack_patterns = {}
for attack_id, description in ATTACK_DESCRIPTIONS.items():
    mask = (df['ATTACK_ID'].values == attack_id)
    if mask.sum() == 0:
        continue
    z = zscore(df.loc[mask, sensor_cols].values, normal_median, normal_mad)
    implicated = top_k_sensors(z, k=5)
    attack_patterns[attack_id] = {
        "attack_id": f"ATTACK_{attack_id}",
        "attack_number": attack_id,
        "description": description,
        "primary_sensor": ATTACK_PRIMARY_SENSOR.get(attack_id),
        "secondary_sensor": ATTACK_SECONDARY_SENSOR.get(attack_id),
        "implicated_sensors": [t for t, _ in implicated],
        "implicated_scores": [s for _, s in implicated],
    }

for a in attack_patterns.values():
    print(a["attack_id"], "->", a["implicated_sensors"])


ATTACK_1 -> ['P3_STATE', 'AIT 503', 'MV 302', 'P301 Status', 'FIT 601']
ATTACK_2 -> ['P3_STATE', 'FIT 601', 'MV 302', 'AIT 503', 'P301 Status']
ATTACK_3 -> ['P3_STATE', 'P101 Status', 'P203 Status', 'P205 Status', 'MV201']
ATTACK_4 -> ['FIT 101', 'AIT 503', 'LSH 601', 'MV 101', 'MV201']
ATTACK_5 -> ['P3_STATE', 'MV 302', 'P301 Status', 'LSH 601', 'MV 501']
ATTACK_6 -> ['P3_STATE', 'MV 302', 'FIT 601', 'AIT 503', 'P301 Status']


## 11. Per-model attribution for the held-out attack

The detection pipeline's forecasters already return `per_node_err` in
section 7-8's `score_forecaster`; `compute_node_errors` below reuses the trained GDN and T-GCN
models to get that breakdown for the flagged windows in the test set (attack 6). Random Forest and
the T-GCN classifier are snapshot models without a forecast target, so their attribution instead
uses raw-value z-scores against the training partition's normal statistics, the same way the
attack-pattern library itself is built, keeping attribution comparable across all four models.

In [16]:
def compute_node_errors(model, edge_or_adj, Xw, batch_size=BATCH_SIZE):
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(Xw, dtype=torch.float32)),
        batch_size=batch_size, shuffle=False)
    model.eval()
    errors = []
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            _, per_node_err = model(xb, edge_or_adj)
            errors.append(per_node_err.cpu().numpy())
    return np.concatenate(errors, axis=0)


def attribute_flagged_windows(scores, threshold, node_errors, k=5):
    """For every window whose score clears the detector's threshold, returns the top-k sensors
    by per-node error in that window."""
    flagged = np.where(np.asarray(scores) >= threshold)[0]
    out = []
    for idx in flagged:
        err = node_errors[idx]
        top_idx = np.argsort(-err)[:k]
        out.append({
            "window_idx": int(idx),
            "implicated_sensors": [sensor_cols[j] for j in top_idx],
            "implicated_scores": [float(err[j]) for j in top_idx],
        })
    return out


gdn_node_errors = compute_node_errors(gdn, edge_index, Xw_test_gdn)
tgcn_node_errors = compute_node_errors(tgcn, adj_norm, Xw_test_tgcn)

gdn_attributions = attribute_flagged_windows(gdn_test_scores, gdn_threshold, gdn_node_errors)
tgcn_attributions = attribute_flagged_windows(tgcn_test_scores, tgcn_threshold, tgcn_node_errors)

# Random Forest and the T-GCN classifier: raw z-score attribution on the flagged rows/windows
rf_flagged_rows = np.where(test_probs_rf >= rf_threshold)[0]
rf_z_test = zscore(df.iloc[test_idx][sensor_cols].values, normal_median, normal_mad)
rf_attributions = [{
    "window_idx": int(i),
    "implicated_sensors": [t for t, _ in top_k_sensors(rf_z_test[i:i + 1], k=5)],
    "implicated_scores": [s for _, s in top_k_sensors(rf_z_test[i:i + 1], k=5)],
} for i in rf_flagged_rows]

tgcn_clf_flagged = np.where(tgcn_clf_test_scores >= tgcn_clf_threshold)[0]
tgcn_clf_z_test = zscore(test_scaled_g[WINDOW_TGCN - 1:], np.zeros(n_nodes), np.ones(n_nodes))
tgcn_clf_attributions = [{
    "window_idx": int(i),
    "implicated_sensors": [t for t, _ in top_k_sensors(np.abs(tgcn_clf_z_test[i:i + 1]), k=5)],
    "implicated_scores": [s for _, s in top_k_sensors(np.abs(tgcn_clf_z_test[i:i + 1]), k=5)],
} for i in tgcn_clf_flagged if i < len(tgcn_clf_z_test)]

print(f"Flagged windows -> RF: {len(rf_attributions)}, GDN: {len(gdn_attributions)}, "
      f"T-GCN forecaster: {len(tgcn_attributions)}, T-GCN classifier: {len(tgcn_clf_attributions)}")


Flagged windows -> RF: 331, GDN: 661, T-GCN forecaster: 667, T-GCN classifier: 166


## 12. Knowledge graph

Three sources of structure feed the graph, each node and relationship carries a `dataset`
property (`"swat"`) so the same Neo4j instance can also hold the BATADAL graph without collisions:

- Sensor nodes, typed by SWaT's own tag convention (flow sensor, level sensor, pump, valve, ...)
  and tagged with their process stage (P1-P6).
- Granger-causal edges between sensors, the single graph trained in section 5.
- Attack-pattern nodes, one per documented attack, linked to their top implicated sensors and to
  the primary/secondary sensor the official documentation names as the attack's actual target.

In [17]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
DATASET = "swat"


def load_graph(tx):
    if RESET_DATASET:
        tx.run("MATCH (n {dataset: $dataset}) DETACH DELETE n", dataset=DATASET)

    for tag in sensor_cols:
        tx.run(
            "MERGE (s:Sensor {tag: $tag, dataset: $dataset}) "
            "SET s.asset_type = $asset_type, s.stage = $stage, s.stage_name = $stage_name",
            tag=tag, dataset=DATASET, asset_type=ASSET_TYPES[tag], stage=ASSET_STAGES[tag],
            stage_name=STAGE_NAMES.get(ASSET_STAGES[tag], "unknown"),
        )

    src_idx, dst_idx = np.nonzero(adj)
    for i, j in zip(src_idx.tolist(), dst_idx.tolist()):
        if i == j:
            continue
        tx.run(
            "MATCH (a:Sensor {tag: $a, dataset: $dataset}), (b:Sensor {tag: $b, dataset: $dataset}) "
            "MERGE (a)-[:GRANGER_CAUSES]->(b)",
            a=sensor_cols[i], b=sensor_cols[j], dataset=DATASET,
        )

    for pattern in attack_patterns.values():
        tx.run(
            "MERGE (p:AttackPattern {attack_id: $attack_id, dataset: $dataset}) "
            "SET p.attack_number = $attack_number, p.description = $description",
            attack_id=pattern["attack_id"], dataset=DATASET,
            attack_number=pattern["attack_number"], description=pattern["description"],
        )
        for tag in pattern["implicated_sensors"]:
            tx.run(
                "MATCH (p:AttackPattern {attack_id: $attack_id, dataset: $dataset}), "
                "(s:Sensor {tag: $tag, dataset: $dataset}) "
                "MERGE (p)-[:IMPLICATES]->(s)",
                attack_id=pattern["attack_id"], dataset=DATASET, tag=tag,
            )
        for tag, rel in ((pattern["primary_sensor"], "PRIMARY_TARGET"), (pattern["secondary_sensor"], "SECONDARY_TARGET")):
            if tag is None:
                continue
            tx.run(
                f"MATCH (p:AttackPattern {{attack_id: $attack_id, dataset: $dataset}}), "
                f"(s:Sensor {{tag: $tag, dataset: $dataset}}) "
                f"MERGE (p)-[:{rel}]->(s)",
                attack_id=pattern["attack_id"], dataset=DATASET, tag=tag,
            )


with driver.session(database=NEO4J_DATABASE) as session:
    session.execute_write(load_graph)
print("Knowledge graph loaded.")


Knowledge graph loaded.


## 13. Retrieval

Given a detector's top implicated sensors for a flagged window, pulls those sensor nodes and their
asset types/stages, Granger-linked neighbours, and matching `AttackPattern` nodes ranked by sensor
overlap. `exclude_attack_id` lets a query skip retrieving its own pattern when explaining a window
that already belongs to a labelled attack, so genuinely unseen retrieval is what gets exercised.

In [18]:
def retrieve_subgraph(implicated_tags, top_k_patterns=2, exclude_attack_id=None):
    with driver.session(database=NEO4J_DATABASE) as session:
        sensors = session.run(
            "MATCH (s:Sensor) WHERE s.tag IN $tags AND s.dataset = $dataset "
            "RETURN s.tag AS tag, s.asset_type AS asset_type, s.stage_name AS stage_name",
            tags=implicated_tags, dataset=DATASET,
        ).data()

        neighbors = session.run(
            "MATCH (s:Sensor)-[r:GRANGER_CAUSES]-(n:Sensor) "
            "WHERE s.tag IN $tags AND s.dataset = $dataset AND n.dataset = $dataset "
            "RETURN DISTINCT s.tag AS from_tag, n.tag AS to_tag, n.asset_type AS to_type, "
            "type(r) AS relationship LIMIT 25",
            tags=implicated_tags, dataset=DATASET,
        ).data()

        pattern_rows = session.run(
            "MATCH (p:AttackPattern)-[:IMPLICATES]->(s:Sensor) "
            "WHERE s.tag IN $tags AND s.dataset = $dataset AND p.dataset = $dataset "
            "RETURN p.attack_id AS attack_id, p.attack_number AS attack_number, "
            "p.description AS description, count(s) AS overlap "
            "ORDER BY overlap DESC LIMIT 5",
            tags=implicated_tags, dataset=DATASET,
        ).data()

    matched_patterns = [
        row for row in pattern_rows
        if exclude_attack_id is None or row["attack_number"] != exclude_attack_id
    ][:top_k_patterns]

    return {"sensors": sensors, "neighbors": neighbors, "matched_patterns": matched_patterns}


## 14. Explanation generation with Groq

In [19]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "openai/gpt-oss-120b"


def build_prompt(model_name, entry, subgraph):
    sensor_lines = "\n".join(
        f"- {tag} (deviation score {z:.1f})"
        for tag, z in zip(entry["implicated_sensors"], entry["implicated_scores"])
    )
    type_lookup = {s["tag"]: (s["asset_type"], s["stage_name"]) for s in subgraph["sensors"]}
    type_lines = "\n".join(
        f"- {tag}: {type_lookup.get(tag, ('unknown', 'unknown'))[0]} in {type_lookup.get(tag, ('unknown', 'unknown'))[1]}"
        for tag in entry["implicated_sensors"]
    )
    neighbor_lines = "\n".join(
        f"- {n['from_tag']} <-> {n['to_tag']} ({n['to_type']}), {n['relationship']}"
        for n in subgraph["neighbors"]
    ) or "(none retrieved)"
    pattern_lines = "\n".join(
        f"- {p['attack_id']}: {p['description']}" for p in subgraph["matched_patterns"]
    ) or "(no similar documented attack retrieved)"

    return (
        f"A {model_name} anomaly detector on the SWaT water-treatment testbed flagged a window as "
        f"anomalous. The sensors with the largest deviation from normal behaviour were:\n{sensor_lines}\n\n"
        f"Asset context for those sensors:\n{type_lines}\n\n"
        f"Granger-causal neighbours of those sensors:\n{neighbor_lines}\n\n"
        f"Most similar documented attacks retrieved from the knowledge graph:\n{pattern_lines}\n\n"
        "In 3-5 sentences, explain in plain engineering language what physically appears to be "
        "happening in the plant, name the implicated sensors and assets explicitly, and say whether "
        "this resembles one of the retrieved documented attacks or looks like a novel pattern. Do not "
        "invent sensor names or attacks that were not given above."
    )


def explain(model_name, entry, exclude_attack_id=None):
    subgraph = retrieve_subgraph(entry["implicated_sensors"], exclude_attack_id=exclude_attack_id)
    prompt = build_prompt(model_name, entry, subgraph)
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return {
        "model_name": model_name, "entry": entry, "subgraph": subgraph,
        "prompt": prompt, "explanation": response.choices[0].message.content,
    }


# Explain a handful of flagged windows per detector, capped so a first run doesn't burn the whole
# Groq free-tier quota
MAX_EXPLANATIONS_PER_MODEL = 5
all_explanations = []
for model_name, attributions in [
    ("Random Forest", rf_attributions), ("GDN", gdn_attributions),
    ("T-GCN forecaster", tgcn_attributions), ("T-GCN classifier", tgcn_clf_attributions),
]:
    for entry in attributions[:MAX_EXPLANATIONS_PER_MODEL]:
        item = explain(model_name, entry, exclude_attack_id=6)  # attack 6 is the test attack itself
        all_explanations.append(item)
        print(f"--- {model_name}, window {entry['window_idx']} ---")
        print(item["explanation"])
        print()


--- Random Forest, window 217 ---
The detector is flagging a sudden, extreme change in the P3 ultrafiltration loop – the pump status (P3_STATE) and its motorised valve (MV 302) are both reporting values far outside normal ranges, while the chemical‑dosing analyzer in P2 (AIT 201) and the ultrafiltration analyzer in P3 (AIT 303) are also showing abnormal readings, and the reverse‑osmosis pressure sensor (PIT 502) is deviating as well. This pattern suggests that the P3 pump may be stuck on or off and the valve may be stuck open or closed, causing a disruption of flow and pressure that propagates to the downstream RO stage, while the analyzers are detecting out‑of‑spec water quality as a consequence. The implicated sensors and assets are **P3_STATE (pump P3), MV 302 (valve P3), AIT 201 (analyzer P2), AIT 303 (analyzer P3), and PIT 502 (pressure sensor P5)**. The observed behaviour does not match the documented attacks (spoofed flow on FIT401 or spoofed level on LIT301) and therefore appea

## 15. Explanation quality check

A lightweight, programmatic proxy for a human qualitative read: does each explanation name-check the
true implicated sensors and asset types it was actually given, and did retrieval surface a genuinely
matching documented attack rather than nothing (or a hallucinated one). This is not a substitute for
the qualitative discussion in the dissertation write-up, but it catches the cheap failure modes
automatically and lets the four detectors be compared on identical grounds.

In [20]:
def check_explanation(item):
    text = item["explanation"].lower()
    entry = item["entry"]
    true_sensors = entry["implicated_sensors"]

    sensor_mentions = sum(1 for t in true_sensors if t.lower() in text)
    matched_ids = {p["attack_id"] for p in item["subgraph"]["matched_patterns"]}

    return {
        "model_name": item["model_name"],
        "window_idx": entry["window_idx"],
        "sensor_mentions": sensor_mentions,
        "n_true_sensors": len(true_sensors),
        "sensor_coverage": sensor_mentions / max(1, len(true_sensors)),
        "n_patterns_retrieved": len(matched_ids),
    }


quality_df = pd.DataFrame([check_explanation(item) for item in all_explanations])
quality_summary = quality_df.groupby("model_name")[["sensor_coverage", "n_patterns_retrieved"]].mean().round(2)
quality_df.to_csv(os.path.join(OUT_DIR, "explanation_quality.csv"), index=False)
quality_summary


,sensor_coverage,n_patterns_retrieved
model_name,,
GDN,0.0,0.8
Random Forest,0.2,2.0
T-GCN classifier,0.0,0.0
T-GCN forecaster,0.0,2.0


In [21]:
import re

def normalize(s):
    return re.sub(r'[\s_]+', '', s.lower())

def check_explanation(item):
    text = item["explanation"].lower()
    entry = item["entry"]
    true_sensors = entry["implicated_sensors"]

    sensor_mentions = sum(1 for t in true_sensors if normalize(t) in normalize(text))
    matched_ids = {p["attack_id"] for p in item["subgraph"]["matched_patterns"]}

    return {
        "model_name": item["model_name"],
        "window_idx": entry["window_idx"],
        "sensor_mentions": sensor_mentions,
        "n_true_sensors": len(true_sensors),
        "sensor_coverage": sensor_mentions / max(1, len(true_sensors)),
        "n_patterns_retrieved": len(matched_ids),
    }

quality_df = pd.DataFrame([check_explanation(item) for item in all_explanations])
quality_summary = quality_df.groupby("model_name")[["sensor_coverage", "n_patterns_retrieved"]].mean().round(2)
quality_df.to_csv(os.path.join(OUT_DIR, "explanation_quality.csv"), index=False)
quality_summary

,sensor_coverage,n_patterns_retrieved
model_name,,
GDN,1.00,0.8
Random Forest,1.00,2.0
T-GCN classifier,1.00,0.0
T-GCN forecaster,0.96,2.0


In [22]:
zero_match = [
    it for it in all_explanations
    if it["model_name"] == "T-GCN classifier" and len(it["subgraph"]["matched_patterns"]) == 0
]

for it in zero_match[:5]:
    idx = it["entry"]["window_idx"]
    print(idx, df.iloc[idx][['ATTACK_ID', 'label']].values, it["entry"]["implicated_sensors"])

0 [np.int64(0) np.int64(0)] ['AIT 303', 'PIT 502', 'PIT 501', 'FIT 401', 'PIT 503']
1 [np.int64(0) np.int64(0)] ['AIT 303', 'PIT 502', 'PIT 501', 'FIT 401', 'PIT 503']
14 [np.int64(0) np.int64(0)] ['AIT 303', 'PIT 502', 'PIT 503', 'PIT 501', 'AIT 201']
15 [np.int64(0) np.int64(0)] ['AIT 303', 'PIT 502', 'PIT 503', 'PIT 501', 'AIT 201']
45 [np.int64(0) np.int64(0)] ['AIT 303', 'PIT 502', 'FIT 503', 'PIT 501', 'FIT 501']


## Notes

Unlike BATADAL, SWaT has only a Granger graph here (no correlation graph, no PLC-level
control-logic edges) and six attacks rather than seven, each targeting a distinct physical
point rather than falling into documented sibling pairs. Retrieval and the explanation-quality
check are therefore evaluated against sensor-level overlap only, not a sibling-attack label.

MAX_EXPLANATIONS_PER_MODEL is kept small here; raise it once Groq quota and Neo4j Aura
free-tier limits are confirmed comfortable for a full run over every flagged window.